In [1]:
# ============================================================
# NOTEBOOK 01 — Data Exploration (Bangladesh)
# Purpose : Initial exploration and learning
#           SAR + Optical flood mapping over Sylhet
# Status  : Reference notebook — code reused in Pakistan study
# Date    : April 2026
# Author  : Newaz Ibrahim Khan
# ============================================================


import ee
import geemap
import torch
import numpy as np

ee.Initialize(project='ee-newazkhn')

print(f"PyTorch  : {torch.__version__}")
print(f"GEE      : connected")
print(f"GPU      : {torch.cuda.is_available()}")

# Quick Sentinel-1 test over Bangladesh
img = (ee.ImageCollection('COPERNICUS/S1_GRD')
         .filterBounds(ee.Geometry.Point([90.4, 23.8]))
         .filterDate('2020-06-01','2020-09-30')
         .first())

print(f"Sentinel-1 bands : {img.bandNames().getInfo()}")
print("\n✓ Everything is working!")

C:\Users\mahmu\anaconda3\envs\floodpinn\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


PyTorch  : 2.11.0+cpu
GEE      : connected
GPU      : False
Sentinel-1 bands : ['VV', 'VH', 'angle']

✓ Everything is working!


In [1]:
import ee
import geemap

ee.Initialize(project='ee-newazkhn')

# Define Sylhet flood region
sylhet = ee.Geometry.Rectangle([91.5, 24.5, 92.5, 25.3])

# Load Sentinel-1 before flood (dry season)
s1_before = (ee.ImageCollection('COPERNICUS/S1_GRD')
  .filterBounds(sylhet)
  .filterDate('2020-02-01', '2020-04-30')
  .filter(ee.Filter.eq('instrumentMode', 'IW'))
  .filter(ee.Filter.listContains(
      'transmitterReceiverPolarisation', 'VV'))
  .select(['VV','VH'])
  .mean())

# Load Sentinel-1 during flood
s1_during = (ee.ImageCollection('COPERNICUS/S1_GRD')
  .filterBounds(sylhet)
  .filterDate('2020-06-15', '2020-09-30')
  .filter(ee.Filter.eq('instrumentMode', 'IW'))
  .filter(ee.Filter.listContains(
      'transmitterReceiverPolarisation', 'VV'))
  .select(['VV','VH'])
  .mean())

# Simple flood detection
flood_mask = s1_during.select('VV').subtract(
             s1_before.select('VV')).lt(-3)

# Display on interactive map
Map = geemap.Map(center=[25.0, 91.9], zoom=9)
Map.addLayer(s1_before.select('VV'),
  {'min':-25,'max':0,'palette':['black','white']},
  'SAR Before Flood')
Map.addLayer(s1_during.select('VV'),
  {'min':-25,'max':0,'palette':['black','white']},
  'SAR During Flood')
Map.addLayer(flood_mask.selfMask(),
  {'palette':['blue']}, 'Flood Extent')
Map.addLayerControl()
Map

C:\Users\mahmu\anaconda3\envs\floodpinn\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Map(center=[25.0, 91.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [3]:
import ee
import geemap

ee.Initialize(project='ee-newazkhn')

# Same Sylhet region
sylhet = ee.Geometry.Rectangle([91.5, 24.5, 92.5, 25.3])

# ── Sentinel-2 Optical ───────────────────────────────────
# Before flood (dry season - clear sky)
s2_before = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(sylhet)
  .filterDate('2020-01-01', '2020-04-30')
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
  .select(['B2','B3','B4','B8','B11','B12'])
  .mean())

# During flood
s2_during = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(sylhet)
  .filterDate('2020-06-15', '2020-09-30')
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
  .select(['B2','B3','B4','B8','B11','B12'])
  .mean())

# ── Compute Water Indices ────────────────────────────────
# NDWI — highlights open water (green + NIR)
ndwi_before = s2_before.normalizedDifference(['B3','B8'])
ndwi_during = s2_during.normalizedDifference(['B3','B8'])

# MNDWI — better for flood (green + SWIR)
mndwi_before = s2_before.normalizedDifference(['B3','B11'])
mndwi_during = s2_during.normalizedDifference(['B3','B11'])

# ── Optical flood mask ───────────────────────────────────
# Water pixels have MNDWI > 0
optical_flood = mndwi_during.gt(0).And(
                mndwi_before.lt(0))

# ── SAR layers (from previous cell) ─────────────────────
s1_before = (ee.ImageCollection('COPERNICUS/S1_GRD')
  .filterBounds(sylhet)
  .filterDate('2020-02-01', '2020-04-30')
  .filter(ee.Filter.eq('instrumentMode', 'IW'))
  .filter(ee.Filter.listContains(
      'transmitterReceiverPolarisation', 'VV'))
  .select(['VV','VH'])
  .mean())

s1_during = (ee.ImageCollection('COPERNICUS/S1_GRD')
  .filterBounds(sylhet)
  .filterDate('2020-06-15', '2020-09-30')
  .filter(ee.Filter.eq('instrumentMode', 'IW'))
  .filter(ee.Filter.listContains(
      'transmitterReceiverPolarisation', 'VV'))
  .select(['VV','VH'])
  .mean())

sar_flood = s1_during.select('VV').subtract(
            s1_before.select('VV')).lt(-3)

# ── Combined SAR + Optical flood ─────────────────────────
# Pixel is flood if EITHER SAR or optical detects it
combined_flood = sar_flood.Or(optical_flood)

# ── Build the comparison map ─────────────────────────────
Map2 = geemap.Map(center=[25.0, 91.9], zoom=9)

# True colour (RGB) before flood
Map2.addLayer(s2_before,
  {'bands':['B4','B3','B2'],
   'min':0, 'max':3000},
  'True Colour BEFORE flood')

# True colour during flood
Map2.addLayer(s2_during,
  {'bands':['B4','B3','B2'],
   'min':0, 'max':3000},
  'True Colour DURING flood')

# False colour (NIR composite) — vegetation shows red
Map2.addLayer(s2_during,
  {'bands':['B8','B4','B3'],
   'min':0, 'max':3000},
  'False Colour DURING flood')

# MNDWI water index
Map2.addLayer(mndwi_during,
  {'min':-0.5, 'max':0.5,
   'palette':['brown','white','cyan']},
  'MNDWI Water Index')

# SAR flood only (blue)
Map2.addLayer(sar_flood.selfMask(),
  {'palette':['0000FF']},
  'SAR flood detection')

# Optical flood only (green)
Map2.addLayer(optical_flood.selfMask(),
  {'palette':['00AA00']},
  'Optical flood detection')

# Combined flood (red = both agree)
Map2.addLayer(combined_flood.selfMask(),
  {'palette':['FF0000']},
  'Combined SAR + Optical flood')

Map2.addLayerControl()
Map2

Map(center=[25.0, 91.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …